# Sentence Similarity Playground (Local Models)

Use this notebook to experiment with locally executed Hugging Face models.

This is helpful when you want to compute embedding vectors encapsulating the semantic meaning of a given text.

In [ ]:
import os
from dotenv import load_dotenv

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
load_dotenv(dotenv_path=".env.local")

In [ ]:
import torch

from sentence_transformers import SentenceTransformer
from rich.console import Console
from rich.table import Table
from typing import List

In [ ]:
sentences: List[str] = [
    "Checkout API returns 500 when coupon code is missing.",
    "Payment endpoint crashes if promo field is null.",
    "Treasury yields fell after softer inflation data.",
    "The central bank signaled caution on future rate cuts.",
    "The overnight train from Tokyo to Kyoto has sleeper cabins.",
    "Fast rail between Tokyo and Kyoto takes about two hours.",
    "A mountain trail near the lake offers quiet sunrise views.",
    "City buses were delayed due to heavy evening rain.",
    "Soft ambient synths create a calm reflective mood.",
    "Atmospheric electronic layers feel introspective and gentle.",
    "Distorted guitars and loud drums drive the chorus.",
    "Jazz improvisation with upright bass filled the small club.",
    "Grilled mushrooms marinated in soy and sesame taste smoky.",
    "The striker scored twice in a dramatic final match."
]

In [ ]:
def get_local_sentence_transformer(
        model_id: str,
        models_folder: str = "./huggingface",
        optimize_dtype: bool = True
) -> SentenceTransformer:
    model_kwargs = None
    if torch.cuda.is_available():
        device = "cuda"
        if optimize_dtype:
            model_kwargs = { "torch_dtype": torch.float16 }
    else:
        device = "cpu"

    print(f"Loading model \"{model_id}\" on {device}")
    transformer = SentenceTransformer(model_id, cache_folder=models_folder, device=device, model_kwargs=model_kwargs)

    return transformer

In [ ]:
def run_sentence_similarity_experiment(model_id: str, optimize_dtype: bool = True):
    transformer = get_local_sentence_transformer(model_id, optimize_dtype=optimize_dtype)

    print(f"Sentence embedding dimension: {transformer.get_sentence_embedding_dimension()}")

    embeddings = transformer.encode(sentences)
    similarity = transformer.similarity(embeddings, embeddings)

    console = Console()

    table = Table(show_header=True, show_lines=True)
    table.add_column("Sentence #1")
    table.add_column("Sentence #2")
    table.add_column("Similarity", justify="right")

    for i in range(len(sentences)):
        for j in range(i + 1, len(sentences)):
            score_val = similarity[i][j]
            if score_val >= 0.5:
                similarity_cell = f"[green]{score_val:.5f}[/green]"
            elif score_val >= 0.25:
                similarity_cell = f"[yellow]{score_val:.5f}[/yellow]"
            else:
                similarity_cell = f"{score_val:.5f}"

            table.add_row(sentences[i], sentences[j], similarity_cell)


    console.print(table)

In [ ]:
run_sentence_similarity_experiment("Qwen/Qwen3-Embedding-0.6B")

In [ ]:
run_sentence_similarity_experiment("BAAI/bge-m3")

In [ ]:
# NOTE: Tony Troeff, 19/03/2026 - When using float16, the similarity computations tend to over/underflow resulting in `NaN`.
run_sentence_similarity_experiment("google/embeddinggemma-300m", optimize_dtype=False)

In [ ]:
run_sentence_similarity_experiment("sentence-transformers/all-MiniLM-L6-v2")